# Notebook 3: Phan tich Xu huong

Xu huong dang bai theo thoi gian.

In [ ]:
import sys, re
from pathlib import Path
from datetime import datetime
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.storage import DataStorage
from config.settings import DB_PATH, RAW_DIR

plt.style.use('seaborn-v0_8-darkgrid')
storage = DataStorage(db_path=DB_PATH, raw_dir=RAW_DIR)
df = storage.load_from_sqlite()
print(f'Tai {len(df)} bai viet')

## 1. Parse ngay dang

In [ ]:
def parse_date(s):
    if not isinstance(s, str) or not s.strip():
        return pd.NaT
    s = re.sub(r'(Th\u01b0\u0301 \w+|GMT[+-]\d+)', '', s).strip()
    for fmt in ['%d/%m/%Y, %H:%M', '%d/%m/%Y %H:%M', '%d-%m-%Y', '%Y-%m-%d']:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return pd.NaT

df['date_parsed'] = df['published_date'].apply(parse_date)
df['date_only'] = df['date_parsed'].dt.date
df['hour'] = df['date_parsed'].dt.hour
df['weekday'] = df['date_parsed'].dt.dayofweek
df['weekday_name'] = df['date_parsed'].dt.day_name()

valid = df.dropna(subset=['date_parsed'])
print(f'Bai co ngay hop le: {len(valid)}/{len(df)}')

## 2. Xu huong theo ngay

In [ ]:
daily = valid.groupby('date_only').size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(daily)), daily['count'], alpha=0.3, color='#667eea')
ax.plot(range(len(daily)), daily['count'], color='#667eea', linewidth=2, marker='o', markersize=4)
ax.set_xticks(range(0, len(daily), max(1, len(daily)//10)))
ax.set_xticklabels([str(d) for d in daily['date_only'].iloc[::max(1, len(daily)//10)]],
                    rotation=45, ha='right')
ax.set_title('So bai viet theo ngay', fontsize=14, fontweight='bold')
ax.set_ylabel('So bai')
plt.tight_layout()
plt.show()

## 3. Phan phoi theo thu trong tuan

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_vi = ['Thu 2','Thu 3','Thu 4','Thu 5','Thu 6','Thu 7','CN']

wd_counts = valid['weekday_name'].value_counts().reindex(weekday_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(weekday_vi, wd_counts.values, color=sns.color_palette('husl', 7))
ax.set_title('Phan phoi bai viet theo thu trong tuan', fontsize=14, fontweight='bold')
ax.set_ylabel('So bai')
for bar, val in zip(bars, wd_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Phan phoi theo gio dang

In [ ]:
hour_counts = valid['hour'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(hour_counts.index, hour_counts.values, color=sns.color_palette('YlOrRd', 24))
ax.set_title('Phan phoi bai viet theo gio', fontsize=14, fontweight='bold')
ax.set_xlabel('Gio (0-23)')
ax.set_ylabel('So bai')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

print(f'Gio dang nhieu nhat: {hour_counts.idxmax()}h ({hour_counts.max()} bai)')

## 5. Xu huong theo thang

In [ ]:
valid['month'] = valid['date_parsed'].dt.to_period('M').astype(str)
monthly = valid.groupby('month').size().reset_index(name='count')

if len(monthly) > 1:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(monthly['month'], monthly['count'], color='#667eea', alpha=0.85)
    ax.set_title('So bai viet theo thang', fontsize=14, fontweight='bold')
    ax.set_ylabel('So bai')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Chua du du lieu nhieu thang. Hay scrape them!')